# Multi-Agent Remittance Processing

This notebook coordinates four specialised agents: **Data Ingestion**, **Remittance Processing**, **Payment Matching**, and **Smart Matching**. The supplied data mimics a bank CSV, an email notice, and ERP invoices.

In [6]:
from dataclasses import asdict
from project_01_remittance_multi_agent import RemittanceOrchestrator, demo_data

bank_csv, email_notice, invoices = demo_data()
workflow = RemittanceOrchestrator()
remittances, decisions = workflow.run(bank_csv, email_notice, invoices)
print(f'Processed {len(remittances)} remittances against {len(invoices)} ERP invoices.')

Processed 4 remittances against 5 ERP invoices.


## Agent hand-off trace

The ingestion and processing agents validate and standardise source data before rules are evaluated. Only unresolved payments reach smart matching.

In [7]:
for payment in remittances:
    print(f'{payment.remittance_id}: source={payment.source}, payer={payment.payer}, amount=${payment.amount:,.2f}, ref={payment.invoice_reference}')

PAY-1001: source=bank_csv, payer=Northwind Traders, amount=$1,250.00, ref=INV-1001
PAY-1002: source=bank_csv, payer=Contoso Retail, amount=$875.50, ref=None
PAY-1003: source=bank_csv, payer=Adventure Works, amount=$1,440.00, ref=INV-1004
PAY-1004: source=email, payer=Fabrikam Ltd, amount=$640.00, ref=INV-1005A


## Matching results

`matched` can be posted automatically under policy. `suggested` remains a human-approved recommendation, so the AI does not post a payment on its own.

In [8]:
import pandas as pd

results = pd.DataFrame([asdict(decision) for decision in decisions])
results[['remittance_id', 'status', 'match_type', 'invoice_ids', 'confidence', 'reason']]

,remittance_id,status,match_type,invoice_ids,confidence,reason
0,PAY-1001,matched,3-way exact,[INV-1001],1.000,"Invoice reference, payer, and amount agree."
1,PAY-1002,matched,2-way exact,[INV-1002],0.960,Payer and amount agree; no reliable invoice re...
2,PAY-1003,matched,3-way exact,[INV-1004],1.000,"Invoice reference, payer, and amount agree."
3,PAY-1004,suggested,AI-assisted fuzzy,[INV-1005],0.917,payer similarity 85%; amount similarity 100%; ...


In [9]:
summary = results.groupby(['status', 'match_type']).size().reset_index(name='payments')
display(summary)
assert len(results) == 4
assert (results['status'] == 'matched').sum() == 3
assert results.loc[results.remittance_id == 'PAY-1004', 'invoice_ids'].iloc[0] == ['INV-1005']
print('✓ Workflow checks passed.')

,status,match_type,payments
0,matched,2-way exact,1
1,matched,3-way exact,2
2,suggested,AI-assisted fuzzy,1


✓ Workflow checks passed.
